In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes pandas torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 25.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━ 334.3/363.4 MB 83.3 MB/s eta 0:00:01m

Model page: https://huggingface.co/meta-llama/Meta-Llama-3-8B

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/meta-llama/Meta-Llama-3-8B)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import json

In [ ]:
import pandas as pd
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    TrainingArguments,
    Trainer
)
import torch

# Load and preview CSV
df = pd.read_csv("/kaggle/input/labels-web-of-law/279_labels_data.csv")  # Assumes columns: 'text', 'label'

df = df.dropna(subset=["text", "label"])  # Remove empty rows if any

# Encode labels
le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"])

# Convert to Hugging Face dataset
hf_dataset = Dataset.from_pandas(df[["text", "label_id"]])
hf_dataset = hf_dataset.train_test_split(test_size=0.1)


In [ ]:
# Model choice — change this to any model you'd like to fine-tune
model_name = "bert-based-uncased"  # e.g. roberta-base, distilbert-base-uncased, etc.

# Load config, tokenizer, model using Auto classes
config = AutoConfig.from_pretrained(model_name, num_labels=len(le.classes_))
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)


In [ ]:
# Tokenization function
def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=512)

tokenized_dataset = hf_dataset.map(tokenize, batched=True)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label_id"])
tokenized_dataset = tokenized_dataset.rename_column("label_id", "labels")

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./model-output",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
)

# Train
trainer.train()